### MBAI 5310G-001: AI Programming
Week 5 Assignment: Unsupervised Learning for Pattern Discovery

### Task 1: Understand the Business Problem
What is the main business problem? Why is customer/user segmentation useful for this business? What kind of marketing decisions could be improved by discovering customer/user groups?

Since the company receives many free sign-ups each month, the sales team cannot give every trial account the same level of attention. The data can help the business prioritize follow-up calls, onboarding support, discount offers, and customer success resources.

### Task 2: Prepare the Dataset

Converted_To_Paid is the target variable, there for is not used to train the K-means model. 

In [1]:
# Load the Dataset

import pandas as pd

# Load the Excel dataset
file_path = "clouddesk_saas_trial_conversion_dataset.csv"
df = pd.read_csv(file_path)

# Display the first five rows
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'clouddesk_saas_trial_conversion_dataset.csv'

In [ ]:
# Check the number of rows and columns
df.shape
# Check column names
df.columns
# Check data types and general information
df.info()

In [ ]:
# Check missing values

df.isnull().sum()

In [ ]:
# Check duplicate rows

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

In [ ]:
# Show summary statistics for numerical columns

df.describe()

### Dataset Inspection

In [ ]:
# Create a copy of the original dataset
df_clean = df.copy()

# Clean column names by removing extra spaces
df_clean.columns = df_clean.columns.str.strip()

# Display original shape
print("Original dataset shape:", df_clean.shape)

In [ ]:
# Remove duplicate rows
df_clean = df_clean.drop_duplicates()

print("Dataset shape after removing duplicates:", df_clean.shape)

In [ ]:
# Check missing values after removing duplicates

df_clean.isnull().sum()

In [ ]:
numerical_columns = [
    "Trial_Length_Days",
    "Login_Count_14_Days",
    "Features_Used",
    "Team_Invites",
    "Pricing_Pages_Viewed",
    "Email_Clicks",
    "Webinar_Attended",
    "Support_Tickets",
    "Discount_Offered_Percent",
    "Competitor_Mentioned",
    "Trial_Completion_Percent",
    "Product_Fit_Score",
    "Estimated_Annual_Value"
]

X = df_clean[numerical_columns]

# Display selected features
X.head()

In [ ]:
print("Features selected for clustering:")
for feature in numerical_columns:
    print("-", feature)

In [ ]:
for col in numerical_columns:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )

    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # Fill missing values after converting to numeric
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
df_clean.isnull().sum()

In [ ]:
X = df_clean[numerical_columns]

# Confirm that there are no missing values in X
X.isnull().sum()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Scale the selected features
X_scaled = scaler.fit_transform(X)

# Convert scaled data back to a DataFrame for better readability
X_scaled_df = pd.DataFrame(X_scaled, columns=numerical_columns)

X_scaled_df.head()

In [ ]:
print("Mean after scaling:")
print(X_scaled_df.mean().round(2))

print()
print("Standard deviation after scaling:")
print(X_scaled_df.std().round(2))

### Task 3: Apply K-means Clustering

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []

K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Finding the Best K")
plt.show()

In [ ]:
elbow_results = pd.DataFrame({
    "K": list(K_range),
    "Inertia": inertia
})

elbow_results

### Analysis of the Elbow Method

I selected K = 3 because the decrease is strongest in the first few cluster values and becomes more gradual after about K = 3.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

cluster_labels = kmeans.fit_predict(X_scaled)

cluster_labels[:10]

In [ ]:
pd.Series(cluster_labels).value_counts().sort_index()

In [ ]:
df_clustered = df_clean.copy()

df_clustered["Cluster"] = cluster_labels

df_clustered.head()

In [ ]:
df_clustered[
    [
        "Trial_ID",
        "Login_Count_14_Days",
        "Features_Used",
        "Pricing_Pages_Viewed",
        "Trial_Completion_Percent",
        "Product_Fit_Score",
        "Estimated_Annual_Value",
        "Converted_To_Paid",
        "Cluster"
    ]
].head(10)

### Task 4: Analyze and Interpret the Clusters

In [ ]:
# Calculate the average value of each feature for each cluster
cluster_summary = df_clustered.groupby("Cluster")[numerical_columns].mean()

cluster_summary_rounded = cluster_summary.round(2)

cluster_summary_rounded

In [ ]:
cluster_counts = df_clustered["Cluster"].value_counts().sort_index()
conversion_rate = df_clustered.groupby("Cluster")["Converted_To_Paid"].mean().round(2)

cluster_summary_final = cluster_summary_rounded.copy()
cluster_summary_final["Number_of_Trial_Accounts"] = cluster_counts
cluster_summary_final["Conversion_Rate_For_Reference"] = conversion_rate

cluster_summary_final

In [ ]:


summary = cluster_summary_final.copy()

low_engagement_cluster = summary["Login_Count_14_Days"].idxmin()

high_value_cluster = summary.drop(index=low_engagement_cluster)["Estimated_Annual_Value"].idxmax()

webinar_fit_cluster = summary.drop(index=[low_engagement_cluster, high_value_cluster]).index[0]

segment_names = {
    high_value_cluster: "High-Value Engaged Trial Accounts",
    low_engagement_cluster: "Low-Engagement Trial Accounts",
    webinar_fit_cluster: "Medium-Engagement Trial Accounts"
}

df_clustered["Segment_Name"] = df_clustered["Cluster"].map(segment_names)

segment_mapping = pd.DataFrame({
    "Cluster": list(segment_names.keys()),
    "Segment_Name": list(segment_names.values())
}).sort_values("Cluster")

segment_mapping

In [3]:
segment_interpretation = pd.DataFrame({
    "Segment Name": [
        "High-Value Engaged Trial Accounts",
        "Low-Engagement Trial Accounts",
        "Medium-Engagement Trial Accounts"
    ],
    "Main Characteristics": [
        "High login activity, more features used, many pricing page views, high trial completion, and the highest estimated annual value.",
        "Lowest login activity, fewer features used, lower pricing interest, lower trial completion, and the lowest conversion rate.",
        "Medium login activity compared with the High-Value Engaged Trial Accounts, similar to the High-Value Engaged Trial Accounts but with the highest product-fit score"
    ],
    "Business Meaning": [
        "This segment is the most valuable, combining strong engagement with the highest revenue potential.",
        "This segment should receive less attention because they have the lowest Product_Fit_Score.",
        "This segment product interest and likely responsiveness to onboarding and sales education."
    ]
})

segment_interpretation

,Segment Name,Main Characteristics,Business Meaning
0,High-Value Engaged Trial Accounts,"High login activity, more features used, many ...","This segment is the most valuable, combining s..."
1,Low-Engagement Trial Accounts,"Lowest login activity, fewer features used, lo...",This segment should receive less attention bec...
2,Medium-Engagement Trial Accounts,Medium login activity compared with the High-V...,This segment product interest and likely respo...


In [ ]:
# Check how many trial accounts are in each named segment

df_clustered["Segment_Name"].value_counts()

In [ ]:
# Calculate average values for each business segment

business_segment_summary = df_clustered.groupby("Segment_Name")[
    [
        "Login_Count_14_Days",
        "Features_Used",
        "Pricing_Pages_Viewed",
        "Trial_Completion_Percent",
        "Product_Fit_Score",
        "Estimated_Annual_Value",
        "Converted_To_Paid"
    ]
].mean().round(2)

business_segment_summary

In [ ]:
# Step 15: Scatter Plot Using Two Selected Features

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.scatter(
    df_clustered["Login_Count_14_Days"],
    df_clustered["Trial_Completion_Percent"],
    c=df_clustered["Cluster"],
    alpha=0.7
)

plt.xlabel("Login Count in First 14 Days")
plt.ylabel("Trial Completion Percent")
plt.title("CloudDesk Trial Segments by Login Activity and Trial Completion")
plt.colorbar(label="Cluster")
plt.show()

### Bar Chart

In [ ]:
metrics_to_compare = [
    "Login_Count_14_Days",
    "Features_Used",
    "Pricing_Pages_Viewed",
    "Trial_Completion_Percent",
    "Product_Fit_Score"
]

segment_avg = df_clustered.groupby("Segment_Name")[metrics_to_compare].mean().round(2)

segment_avg.plot(kind="bar", figsize=(10, 6))
plt.xlabel("Customer/User Segment")
plt.ylabel("Average Value")
plt.title("Average Trial Engagement Metrics by Segment")
plt.xticks(rotation=30, ha="right")
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

### Box Plot 

In [ ]:
segment_order = [
    "High-Value Engaged Trial Accounts",
    "Medium-Engagement Trial Accounts",
    "Low-Engagement Trial Accounts"
]

boxplot_data = [
    df_clustered[df_clustered["Segment_Name"] == segment]["Estimated_Annual_Value"]
    for segment in segment_order
]

plt.figure(figsize=(10, 6))
plt.boxplot(boxplot_data, tick_labels=segment_order)
plt.xlabel("Customer/User Segment")
plt.ylabel("Estimated Annual Value")
plt.title("Estimated Annual Value by Segment")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:

from sklearn.decomposition import PCA

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

df_clustered["PCA1"] = X_pca[:, 0]
df_clustered["PCA2"] = X_pca[:, 1]

df_clustered[["Trial_ID", "Cluster", "Segment_Name", "PCA1", "PCA2"]].head()

### PCA Visualization

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    df_clustered["PCA1"],
    df_clustered["PCA2"],
    c=df_clustered["Cluster"],
    alpha=0.7
)

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("CloudDesk Trial Segments Visualized with PCA")
plt.colorbar(label="Cluster")
plt.show()

In [ ]:
explained_variance = pca.explained_variance_ratio_

print("Explained variance by PCA1:", round(explained_variance[0], 3))
print("Explained variance by PCA2:", round(explained_variance[1], 3))
print("Total explained variance:", round(explained_variance.sum(), 3))

### Task 6: Business Interpretation and Recommendation

High-Value Engaged Trial Accounts and Medium-Engagement Trial Accounts should receive the most direct sales attention because they show strong engagement and high estimated annual value.

Low-Engagement Trial Accounts should receive less attention because they have the lowest Product_Fit_Score.

This analysis can help CloudDesk make better decisions by prioritizing accounts, creating segment-specific campaigns, forecasting conversion potential, and allocating customer success resources more efficiently.

### Task 7: Limitations and Responsible AI Reflection



#### What is one limitation of the dataset or model?
The dataset may not include all real sales factors such as budget timing, buying committees, or relationship quality. 
#### Why does K-means not always produce perfect customer segments?
Real customers may overlap across groups, so cluster boundaries are not perfect business rules. In this cause High-Value Engaged Trial Accounts and Medium-Engagement Trial Accounts overlap a lot.
#### What unfair decision could happen without human review?
Trial accounts with low engagement may be judged as low-value customers by the system, resulting in fewer sales follow-ups, discounts, or customer support. However, some customers may simply need more time to learn about the product or may not be actively using it due to a lack of training.
#### Why should human judgment still be used?
Human judgment is needed to review customer context, avoid unfair treatment, and make practical marketing decisions that a model cannot fully understand.